# **Model**

# **XRF milk database**

In [4]:
# Importing the necessary libraries
import pandas as pd
import numpy as np
import kennard_stone as ks
pd.options.plotting.backend = 'plotly'  # setting plotly as the backend for pandas plotting

# Add parent directory to sys.path so local module 'synthetic' (one level up) can be imported
import sys
from pathlib import Path # for path manipulations
parent_dir = Path.cwd().parent.parent.resolve() # move two levels up from current working directory
if str(parent_dir) not in sys.path: # check to avoid duplicates
    sys.path.insert(0, str(parent_dir)) # insert at the start of sys.path to prioritize local modules

# Loading a soil spectral dataset based on X-ray fluorescence (XRF)
data_complete = pd.read_csv(f'{parent_dir}/XRF_databases/ecigar/plsda/ecigar.csv', sep=';') # local copy of Toledo 2022 dataset
data = data_complete.loc[:, '0.997':'20.007']
#data.insert(0, 'exCa', data_complete['exCa'])  # inserting the target variable (e.g., exCa (exchangeable calcium))

# Creating a new column 'Class' based on the condition of the samples in the 'Type' column being 'Authentic'
data_A = data_complete[data_complete['Class'] == 'A'].reset_index(drop=True)
data_B = data_complete[data_complete['Class'] == 'B'].reset_index(drop=True)

# splitting the data into calibration and prediction sets by kennard-stone algorithm
XA_cal, XA_pred = ks.train_test_split(data_A.loc[:, '0.997':'20.007'], test_size=0.30) # class A
XA_cal = XA_cal.reset_index(drop=True)
XA_pred = XA_pred.reset_index(drop=True)

XB_cal, XB_pred = ks.train_test_split(data_B.loc[:, '0.997':'20.007'], test_size=0.30) # class B
XB_cal = XB_cal.reset_index(drop=True)
XB_pred = XB_pred.reset_index(drop=True)

Xcalclass = pd.concat([XA_cal, XB_cal], axis=0).reset_index(drop=True) # concatenating both classes
Xpredclass = pd.concat([XA_pred, XB_pred], axis=0).reset_index(drop=True)
ycalclass = pd.Series(['A']*XA_cal.shape[0] + ['B']*XB_cal.shape[0]) # creating the target variable for calibration set
ypredclass = pd.Series(['A']*XA_pred.shape[0] + ['B']*XB_pred.shape[0]) # creating the target variable for prediction set

# preprocessings
import preprocessings as prepr # preprocessing methods for XRF data

Xcalclass_prep, mean_calclass, mean_calclass_poisson  = prepr.poisson(Xcalclass, mc=True)
Xpredclass_prep = ((Xpredclass/np.sqrt(mean_calclass)) - mean_calclass_poisson)

from modeling import pls_optimized

# performing PLS-DA with optimized latent variables
plsda_results = pls_optimized(Xcalclass_prep, 
                              ycalclass,
                              LVmax=3,
                              Xpred=Xpredclass_prep,
                              ypred=ypredclass,
                              aim='classification',
                              cv=10)
plsda_results[0]

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
2026-01-29 09:26:43,092 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
2026-01-29 09:26:43,099 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: Futur

,LVs,Accuracy Cal,Sensitivity Cal,Specificity Cal,CM Cal,Accuracy CV,Sensitivity CV,Specificity CV,CM CV,Accuracy Pred,Sensitivity Pred,Specificity Pred,CM Pred,X Cum Exp Var,Y Cum Exp Var,X Ind Exp Var,Y Ind Exp Var
0,1,0.68,0.333333,0.8750,"[[14, 2], [6, 3]]",0.64,0.222222,0.875,"[[14, 2], [7, 2]]",0.545455,0.00,0.857143,"[[6, 1], [4, 0]]",63.224546,7.669394,63.224546,7.669394
1,2,0.80,0.555556,0.9375,"[[15, 1], [4, 5]]",0.56,0.222222,0.750,"[[12, 4], [7, 2]]",0.818182,0.50,1.000000,"[[7, 0], [2, 2]]",69.628977,19.133399,6.404431,11.464005
2,3,0.84,0.777778,0.8750,"[[14, 2], [2, 7]]",0.56,0.222222,0.750,"[[12, 4], [7, 2]]",0.909091,0.75,1.000000,"[[7, 0], [1, 3]]",88.071593,22.397397,18.442616,3.263998


In [8]:
from modeling import svm_optimized

svm_model = svm_optimized(Xcalclass_prep, ycalclass, Xpredclass_prep, ypredclass, aim='classification', kernel='linear')
svm_model[0]


,Model,Accuracy Cal,Sensitivity Cal,Specificity Cal,CM Cal,Accuracy Pred,Sensitivity Pred,Specificity Pred,CM Pred
0,SVC,1.0,1.0,1.0,"[[16, 0], [0, 9]]",0.818182,1.0,0.714286,"[[5, 2], [0, 4]]"


In [9]:
# calculando a covariancia entre cada variável espectral e a predição do modelo SVM
cov_scores = []
y_pred = svm_model[4]['SVC'].values # using the continuous predictions from SVM, extracting as 1D array

for col in Xcalclass_prep.columns:
    x_values = Xcalclass_prep[col].values
    covariance = np.cov(x_values, y_pred)[0, 1] # covariance between x and y
    cov_scores.append(covariance)
cov_scores_df = pd.DataFrame(cov_scores, index=Xcalclass_prep.columns, columns=['Covariance'])
cov_scores_df = np.abs(cov_scores_df)
cov_scores_df.plot()

In [10]:
X_sv = svm_model[3].support_vectors_            # shape (n_SV, n_features)
alpha_dual = svm_model[3].dual_coef_.ravel()    # shape (n_SV,)

# Calculando os coeficientes p usando os vetores de suporte e os multiplicadores de Lagrange
pvetor = pd.DataFrame({'energia' : Xcalclass.columns,
                       'importance': (X_sv.T) @ alpha_dual})
pvetor['importance'] = np.abs(pvetor['importance'])
pvetor['importance'].plot()

In [18]:
# establishing spectral cuts based on expert knowledge of XRF spectra
spectral_cuts = [
('Si', 0.997, 1.920),
('background1', 1.920, 2.413),
('Cl', 2.413, 2.984),
('background2', 2.984, 3.134),
('K', 3.134, 3.479),
('Ca ka', 3.134, 3.839),
('Ca kb', 3.839, 4.104),
('Ti', 4.104, 4.823),
('background3', 4.823, 6.161),
('Fe ka', 6.161, 6.732),
('Fe kb', 6.732, 7.242),
('Ni', 7.242, 7.712),
('Cu', 7.712, 8.348),
('Zn ka', 8.348, 8.963),
('Ga', 8.963, 9.414),
('Zn kb', 9.414, 9.739),
('background4', 9.739, 10.379),
('Pb La', 10.379, 10.725),
('background5', 10.725, 12.391),
('Pb Lb', 12.391, 12.831),
('background6', 12.831, 15.333),
('Mo compton scattering', 15.333, 17.215),
('Mo rayleigh scattering', 17.215, 17.765),
('background7', 17.765, 20.007),
]

In [19]:
import explaining as exp
spectral_zones_class = exp.extract_spectral_zones(Xcalclass_prep, spectral_cuts)
zone_sums_df = exp.aggregate_spectral_zones(spectral_zones_class, aggregator='extreme')
predicates_quantiles = exp.predicates_by_quantiles(zone_sums_df, [0.2, 0.4, 0.6, 0.8])
co_occurrence_matrix_df = predicates_quantiles[2]
predicate_info_dict = exp.create_predicate_info_dict(
    predicates_df=predicates_quantiles[0],
    predicate_indicator_df=predicates_quantiles[1],
    zone_aggregated_df=zone_sums_df,
    y_predicted_numeric=y_pred
)

## Pvector and SHAP

In [20]:
# VIP scores por energia
pvector_df = pd.DataFrame({
    'energy': pvetor['energia'],
    'Pvector': pvetor['importance'].values
})
pvector_df = pvector_df.sort_values(by='Pvector', ascending=False).reset_index(drop=True)
energy_to_zone_vip = {}
for zone_name, start, end in spectral_cuts:
    for e in pvector_df['energy']:
        ef = float(e)
        if start <= ef <= end:
            energy_to_zone_vip[e] = zone_name
pvector_df['Zone'] = pvector_df['energy'].map(energy_to_zone_vip)
pvector_unique_df = pvector_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
pvector_unique_df = pvector_unique_df.sort_values(by='Pvector', ascending=False).reset_index(drop=True)
pvector_unique_df
#shap_unique_df = pd.read_csv('shap_ecigar.csv', sep=';') # loading previously saved shap_unique_df

,energy,Pvector,Zone
0,2.013,0.003362,background1
1,9.214,0.003335,Ga
2,14.132,0.002793,background6
3,17.535,0.002475,Mo rayleigh scattering
4,4.5,0.002465,Ti
5,20.007,0.002194,background7
6,11.886,0.002009,background5
7,6.346,0.001985,Fe ka
8,2.959,0.001923,Cl
9,17.21,0.001835,Mo compton scattering


# permutation

In [21]:
# Permutation importance baseado em mudança nas probabilidades previstas (predict_proba)
# Medimos a média da diferença absoluta entre as probabilidades originais e as probabilidades
# obtidas após permutar cada variável. Isso fornece uma métrica contínua de importância.
n_repeats = 10
rng = np.random.RandomState(42)
# Probabilidades base (classe 'B' mapeada para 1)
baseline_proba = svm_model[3].predict_proba(Xcalclass_prep)[:, 1]
importance_list = []
X_arr = Xcalclass_prep.copy()
for col in Xcalclass_prep.columns:
    diffs = []
    for _ in range(n_repeats):
        X_perm = X_arr.copy()
        X_perm[col] = rng.permutation(X_perm[col].values)
        perm_proba = svm_model[3].predict_proba(X_perm)[:, 1]
        diffs.append(np.mean(np.abs(baseline_proba - perm_proba)))
    importance_list.append(np.mean(diffs))

permutation_df = pd.DataFrame({
    'energy': Xcalclass_prep.columns,
    'Permutation_importance_proba': importance_list
})
permutation_df.sort_values(by='Permutation_importance_proba', ascending=False, inplace=True)
energy_to_zone_vip = {}
for zone_name, start, end in spectral_cuts:
    for e in permutation_df['energy']:
        ef = float(e)
        if start <= ef <= end:
            energy_to_zone_vip[e] = zone_name
permutation_df['Zone'] = permutation_df['energy'].map(energy_to_zone_vip)
permutation_unique_df = permutation_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
permutation_unique_df = permutation_unique_df.sort_values(by='Permutation_importance_proba', ascending=False)
permutation_unique_df

,energy,Permutation_importance_proba,Zone
0,2.663,0.002749,Cl
1,17.535,0.002327,Mo rayleigh scattering
2,17.21,0.001798,Mo compton scattering
3,1.753,0.001540,Si
4,3.324,0.001254,Ca ka
5,2.008,0.000809,background1
6,6.371,0.000703,Fe ka
7,9.214,0.000703,Ga
8,8.623,0.000567,Zn ka
9,17.8,0.000305,background7


In [22]:
permutation_unique_df.to_csv('permutation_ecigar.csv', index=False, sep=';')

# **bagging - covariance**

In [23]:
import explaining as exp

# LISTA DE SEMENTES A TESTAR
random_seeds = [0, 1, 2, 3]

all_results_cov = {}
training_samples = len(Xcalclass)

# LOOP: PROCESSAR CADA SEMENTE
y_predicted_numeric = pd.Series(y_pred) # predições numéricas do modelo

for seed in random_seeds:
    print(f"\n{'='*70}")
    print(f"Processando semente: {seed}")
    print(f"{'='*70}\n")
    # Bagging
    bags_result_seed = exp.bagging_predicates(
        zone_sums_df=zone_sums_df,
        y_predicted_numeric=y_predicted_numeric,
        predicates_df=predicates_quantiles[0],
        n_bags=10,
        #n_predicates_per_bag=40,
        n_samples_per_bag=int(training_samples*0.8), # 80 % da base para amostrar (convertido para int)
        min_samples_per_predicate=int(training_samples*0.2), # 20 % da base para limitar (convertido para int)
        replace=False,
        sample_bagging=True,
        predicate_bagging=False,
        random_seed=seed
    )
    
    # Inserir classe prevista
    for bag_name, pred_dict in bags_result_seed.items(): # iterando sobre cada bag
        for pred_rule, df_info in pred_dict.items():
            df_info['Class_Predicted'] = np.where(df_info['Predicted_Y'] >= 0.5, 'A', 'B') # binarizando com threshold 0.5, A = eut, B = dist
    
    # Calcular MI
    cov_results_dict_seed = exp.calculate_predicate_metrics(
        bags_result=bags_result_seed,
        metric='covariance', # covariance ou mutual_information
        threshold=0.01, # threshold para cortar predicados irrelevantes
        n_neighbors=5
    )
    
    # Salvar no dicionário principal
    all_results_cov[seed] = {
        'bags_result': bags_result_seed,
        'cov_results_dict': cov_results_dict_seed
    }

# CONSTRUÇÃO DE GRAFOS PARA MÚLTIPLAS SEMENTES (LOOP EXTERNO)
# Dicionário para armazenar grafos
graphs_by_seed = {}

for seed in random_seeds:
    print(f"\n{'='*70}")
    print(f"Processando Grafo - Semente: {seed}")
    print(f"{'='*70}\n")
    # Construir grafo para esta semente
    DG = exp.build_predicate_graphv2(
        bags_result=all_results_cov[seed]['bags_result'],
        predicate_ranking_dict=all_results_cov[seed]['cov_results_dict'],
        metric_column='Covariance',  # ou 'Covariance' se mudar a métrica
        random_state=seed,
        show_details=True
    )
    # Armazenar grafo
    graphs_by_seed[seed] = DG

# Calcular LRC usando a função pronta do explaining.py
lrc_cov_by_seed = {}
for seed in random_seeds:
    DG = graphs_by_seed[seed]
    lrc_cov_df_seed = exp.calculate_lrc_single_graph(DG, predicates_quantiles[0])
    lrc_cov_df_seed['Seed'] = seed  # Adicionar coluna com a semente
    lrc_cov_by_seed[seed] = lrc_cov_df_seed

# junando todas as colunas 'Node' de lrc_by_seed em um único dataframe
lrc_cov_all_seeds_df = pd.DataFrame()
for seed in random_seeds:
    lrc_cov_df_seed = lrc_cov_by_seed[seed].rename(columns={'Node': f'Predicate_Cov_Seed_{seed}'})
    lrc_cov_all_seeds_df = pd.concat([lrc_cov_all_seeds_df, lrc_cov_df_seed[[f'Predicate_Cov_Seed_{seed}']]], axis=1)

# vamos filtrar lrc_by_seed em cada semente para manter apenas as zonas espectrais únicas com maior LRC em um mesmo dataframe
lrc_cov_unique_by_seed = {}
for seed, lrc_df in lrc_cov_by_seed.items():
    lrc_cov_unique_df = lrc_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
    lrc_cov_unique_df = lrc_cov_unique_df.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
    lrc_cov_unique_by_seed[seed] = lrc_cov_unique_df

lrc_cov_all_seeds_df # exibindo o dataframe consolidado com predicados de todas as sementes


Processando semente: 0

Bag_1 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 156 | Descartados: 36
Bag_2 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 166 | Descartados: 26
Bag_3 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 158 | Descartados: 34
Bag_4 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 154 | Descartados: 38
Bag_5 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 177 | Descartados: 15
Bag_6 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 144 | Descartados: 48
Bag_7 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 165 | Descartados: 27
Bag_8 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 152 | Descartados: 40
Bag_9 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 153 | Descartados: 39
Bag_10 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 164 | Descartados: 28
Calculando Covariance para Predicados
Métrica: covariance
Threshold: 0.01

Processando semente: 1

Bag_1 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 165 | 

,Predicate_Cov_Seed_0,Predicate_Cov_Seed_1,Predicate_Cov_Seed_2,Predicate_Cov_Seed_3
0,Cu > -14.20,Cu > -14.20,Cu > -14.20,Cu > -16.69
1,Cu > -16.69,Cu > -16.69,Cu > -16.69,Cu > -14.20
2,background7 > 17.05,Ni > -4.20,Zn ka <= -19.93,Cl > -45.57
3,Cl > -45.57,Cu > -17.55,background2 > 5.59,background7 > -11.83
4,Cu > -17.55,Ni > -8.36,Cu > -17.55,Cu > -17.55
...,...,...,...,...
167,background3 > 4.89,Ca kb <= -0.56,Class_A,Ca kb <= -0.56
168,Ca ka <= -41.92,background5 <= -4.50,Class_B,Pb Lb <= -5.92
169,K <= -41.92,background4 <= -4.33,NaN,Pb La <= -5.66
170,Class_A,Class_A,NaN,Class_A


In [24]:
# Somar LRCs de predicados equivalentes entre diferentes seeds
lrc_combined_list = []

for seed in random_seeds:
    lrc_df = lrc_cov_by_seed[seed].copy()
    lrc_combined_list.append(lrc_df) # o append adiciona o dataframe ao final da lista

# Concatenar todos os dataframes
lrc_all_seeds = pd.concat(lrc_combined_list, ignore_index=True) 

# Agrupar por predicado (Node) e somar as LRCs, mantendo Zone, Threshold e Operator
lrc_summed_df = lrc_all_seeds.groupby('Node').agg({ # o .agg pode ser usado para aplicar múltiplas funções de agregação
    'Local_Reaching_Centrality': 'mean', # sum = somando as LRCs, poderia ser média ou outro agregado
    'Zone': 'first',
    'Threshold': 'first',
    'Operator': 'first'
}).reset_index()

# Ordenar pelo valor de LRC somado (maior para menor)
lrc_summed_df = lrc_summed_df.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
# vamoss pegar so os valore sunicos de lrc_summed_df baseado na zona espectral
lrc_summed_unique_df_cov = lrc_summed_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
lrc_summed_unique_df_cov = lrc_summed_unique_df_cov.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
lrc_summed_unique_df_cov

,Node,Local_Reaching_Centrality,Zone,Threshold,Operator
0,Cu > -14.20,22.037461,Cu,-14.20,>
1,Cl > -45.57,6.818997,Cl,-45.57,>
2,background7 > 17.05,5.838552,background7,17.05,>
3,Ni > -4.20,5.147954,Ni,-4.20,>
4,Mo rayleigh scattering > -49.79,4.356249,Mo rayleigh scattering,-49.79,>
5,Zn ka > -24.20,3.750557,Zn ka,-24.20,>
6,Mo compton scattering > -2.19,3.457900,Mo compton scattering,-2.19,>
7,background3 > -6.23,3.344926,background3,-6.23,>
8,Ca ka > -41.92,3.284965,Ca ka,-41.92,>
9,Si > 44.53,3.236779,Si,44.53,>


# **Perturbation**

In [25]:
import explaining as exp

# LISTA DE SEMENTES A TESTAR
random_seeds = [0, 1, 2, 3]

all_results_pert = {}
training_samples = len(Xcalclass)

# LOOP: PROCESSAR CADA SEMENTE
y_predicted_numeric = pd.Series(y_pred) # predições numéricas do modelo

for seed in random_seeds:
    print(f"\n{'='*70}")
    print(f"Processando semente: {seed}")
    print(f"{'='*70}\n")
    # Bagging
    bags_result_seed = exp.bagging_predicates(
        zone_sums_df=zone_sums_df,
        y_predicted_numeric=y_predicted_numeric,
        predicates_df=predicates_quantiles[0],
        n_bags=10,
        #n_predicates_per_bag=40,
        n_samples_per_bag=int(training_samples*0.8), # 80 % da base para amostrar (convertido para int)
        min_samples_per_predicate=int(training_samples*0.2), # 20 % da base para limitar (convertido para int)
        replace=False,
        sample_bagging=True,
        predicate_bagging=False,
        random_seed=seed
    )
    # Inserir classe prevista
    for bag_name, pred_dict in bags_result_seed.items(): # iterando sobre cada bag
        for pred_rule, df_info in pred_dict.items():
            df_info['Class_Predicted'] = np.where(df_info['Predicted_Y'] >= 0.5, 'A', 'B') # binarizando com threshold 0.5, A = eut, B = dist

    pert_results_seed = exp.calculate_predicate_perturbation(
        estimator=svm_model[3],
        Xcalclass_prep=Xcalclass_prep,
        folds_struct=bags_result_seed,
        predicates_df=predicates_quantiles[0],
        spectral_cuts=spectral_cuts,
        #perturbation_value=0,
        perturbation_mode='mean', # valores entre 'mean' ou 'min'
        stats_source='full', # full indica usar todas as amostras para calcular estatísticas enquanto que 'fold' usa apenas as amostras do fold atual
        #metric='mean_abs_diff',   # Média com sinal (pode ser negativo)
        aim='classification',
        metric='probability_shift', 
        verbose=True
    )

    # Remove todos os valores iguais a zero de todos os bags em perm_results[bag]["Permutation"] e salva como perm_results_thresholded
    # pert_results_seed_thresholded = {}
    # for bag, df in perm_results_seed.items():
    #     # Verifica se é um DataFrame e se a coluna 'Permutation' existe
    #     if isinstance(df, pd.DataFrame) and 'Permutation' in df.columns:
    #         filtered_df = df[df['Permutation'] > 0].copy()
    #         pert_results_seed_thresholded[bag] = filtered_df
    #     else:
    #         # Se não for DataFrame esperado, apenas copia
    #         pert_results_seed_thresholded[bag] = df

    # Salvar no dicionário principal
    all_results_pert[seed] = {
        'bags_result': bags_result_seed,
        'pert_results_dict': pert_results_seed
    }

# CONSTRUÇÃO DE GRAFOS PARA MÚLTIPLAS SEMENTES (LOOP EXTERNO)
# Dicionário para armazenar grafos
graphs_pert_by_seed = {}

for seed in random_seeds:
    print(f"\n{'='*70}")
    print(f"Processando Grafo - Semente: {seed}")
    print(f"{'='*70}\n")
    # Construir grafo para esta semente
    DG = exp.build_predicate_graphv2(
        bags_result=all_results_pert[seed]['bags_result'],
        predicate_ranking_dict=all_results_pert[seed]['pert_results_dict'],
        metric_column='Perturbation',  # ou 'Covariance' se mudar a métrica
        random_state=seed,
        show_details=True
    )
    # Armazenar grafo
    graphs_pert_by_seed[seed] = DG  

# Calcular LRC usando a função pronta do explaining.py
lrc_pert_by_seed = {}
for seed in random_seeds:
    DG = graphs_pert_by_seed[seed]
    lrc_pert_df_seed = exp.calculate_lrc_single_graph(DG, predicates_quantiles[0])
    lrc_pert_df_seed['Seed'] = seed  # Adicionar coluna com a semente
    lrc_pert_by_seed[seed] = lrc_pert_df_seed

# junando todas as colunas 'Node' de lrc_by_seed em um único dataframe
lrc_pert_all_seeds_df = pd.DataFrame()
for seed in random_seeds:
    lrc_pert_df_seed = lrc_pert_by_seed[seed].rename(columns={'Node': f'Predicate_pert_Seed_{seed}'})
    lrc_pert_all_seeds_df = pd.concat([lrc_pert_all_seeds_df, lrc_pert_df_seed[[f'Predicate_pert_Seed_{seed}']]], axis=1)

# vamos filtrar lrc_by_seed em cada semente para manter apenas as zonas espectrais únicas com maior LRC em um mesmo dataframe
lrc_pert_unique_by_seed = {}
for seed, lrc_df in lrc_pert_by_seed.items():
    lrc_pert_unique_df = lrc_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
    lrc_pert_unique_df = lrc_pert_unique_df.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
    lrc_pert_unique_by_seed[seed] = lrc_pert_unique_df

lrc_pert_all_seeds_df # exibindo o dataframe consolidado com predicados de todas as sementes


Processando semente: 0

Bag_1 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 156 | Descartados: 36
Bag_2 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 166 | Descartados: 26
Bag_3 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 158 | Descartados: 34
Bag_4 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 154 | Descartados: 38
Bag_5 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 177 | Descartados: 15
Bag_6 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 144 | Descartados: 48
Bag_7 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 165 | Descartados: 27
Bag_8 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 152 | Descartados: 40
Bag_9 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 153 | Descartados: 39
Bag_10 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 164 | Descartados: 28
PERTURBATION IMPORTANCE PARA PREDICADOS
Tipo de tarefa (aim): classification
Modo de perturbação: mean
Fonte das estatísticas: full
Métrica: probability_shift
Total

,Predicate_pert_Seed_0,Predicate_pert_Seed_1,Predicate_pert_Seed_2,Predicate_pert_Seed_3
0,Fe ka > 10.08,K > 48.39,K > 48.39,K > 48.39
1,K > -20.13,background1 > 33.08,background1 > 33.08,Ca ka > 48.39
2,K > 48.39,Ca ka > 48.39,background7 > 17.05,Ca ka > -20.13
3,K > -38.25,background7 > 17.05,Ca ka > 48.39,background7 > 17.05
4,background1 > 33.08,K > -20.13,K > -20.13,background1 > 33.08
...,...,...,...,...
189,Zn kb <= -10.75,Zn kb > -8.99,Pb Lb <= 6.53,Zn kb <= -10.75
190,Zn kb <= -7.43,Zn kb <= 8.42,Zn kb <= -7.43,Zn kb <= -7.43
191,Zn kb <= 8.42,Class_A,Class_A,Zn kb <= 8.42
192,Class_A,Class_B,Class_B,Class_A


In [26]:
all_results_pert[0]['pert_results_dict']['Bag_9']

,Predicate,Perturbation
0,background7 > 17.05,0.023601
1,Fe ka > 10.08,0.021430
2,K > -20.13,0.021031
3,Ca ka > -20.13,0.018558
4,background7 > 7.87,0.018384
...,...,...
148,Zn kb > -8.99,0.000293
149,Zn kb <= -8.99,0.000286
150,Zn kb > -10.75,0.000281
151,Zn kb <= -7.43,0.000224


In [27]:
# Somar LRCs de predicados equivalentes entre diferentes seeds
lrc_combined_list = []

for seed in random_seeds:
    lrc_df = lrc_pert_by_seed[seed].copy()
    lrc_combined_list.append(lrc_df) # o append adiciona o dataframe ao final da lista

# Concatenar todos os dataframes
lrc_all_seeds = pd.concat(lrc_combined_list, ignore_index=True) 

# Agrupar por predicado (Node) e somar as LRCs, mantendo Zone, Threshold e Operator
lrc_summed_df = lrc_all_seeds.groupby('Node').agg({ # o .agg pode ser usado para aplicar múltiplas funções de agregação
    'Local_Reaching_Centrality': 'mean', # sum = somando as LRCs, poderia ser média ou outro agregado
    'Zone': 'first',
    'Threshold': 'first',
    'Operator': 'first'
}).reset_index()

# Ordenar pelo valor de LRC somado (maior para menor)
lrc_summed_df = lrc_summed_df.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
# vamoss pegar so os valore sunicos de lrc_summed_df baseado na zona espectral
lrc_summed_unique_df_pert = lrc_summed_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
lrc_summed_unique_df_pert = lrc_summed_unique_df_pert.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
lrc_summed_unique_df_pert

,Node,Local_Reaching_Centrality,Zone,Threshold,Operator
0,K > 48.39,5.524792,K,48.39,>
1,background1 > 33.08,4.843951,background1,33.08,>
2,Ca ka > 48.39,4.715373,Ca ka,48.39,>
3,background7 > 17.05,4.633477,background7,17.05,>
4,Fe ka > 10.08,3.990992,Fe ka,10.08,>
5,Cl > 94.61,3.871913,Cl,94.61,>
6,Mo compton scattering > 86.16,3.460642,Mo compton scattering,86.16,>
7,Zn ka > 17.61,2.123206,Zn ka,17.61,>
8,Si > 44.53,2.098843,Si,44.53,>
9,Ga <= -13.34,1.989977,Ga,-13.34,<=


In [28]:
import numpy as np

max_len = max(
    len(pvector_unique_df['Zone']),
    #len(shap_unique_df['Zone']),
    len(permutation_unique_df['Zone']),
    len(lrc_summed_unique_df_pert['Zone']),
    len(lrc_summed_unique_df_cov['Zone'])
)

def pad_list(lst, length):
    return list(lst) + [None] * (length - len(lst))

features_importance = pd.DataFrame({
    'SVM_pvector': pad_list(pvector_unique_df['Zone'], max_len),
    #'Shap': pad_list(shap_unique_df['Zone'], max_len),
    'Permutation': pad_list(permutation_unique_df['Zone'], max_len),
    'LRC_perturbation' : pad_list(lrc_summed_unique_df_pert['Zone'], max_len),
    'LRC_covariance' : pad_list(lrc_summed_unique_df_cov['Zone'], max_len),
})

features_importance.to_csv('feature_importance.csv', index=False, sep=';')
features_importance

,SVM_pvector,Permutation,LRC_perturbation,LRC_covariance
0,background1,Cl,K,Cu
1,Ga,Mo rayleigh scattering,background1,Cl
2,background6,Mo compton scattering,Ca ka,background7
3,Mo rayleigh scattering,Si,background7,Ni
4,Ti,Ca ka,Fe ka,Mo rayleigh scattering
5,background7,background1,Cl,Zn ka
6,background5,Fe ka,Mo compton scattering,Mo compton scattering
7,Fe ka,Ga,Zn ka,background3
8,Cl,Zn ka,Si,Ca ka
9,Mo compton scattering,background7,Ga,Si


In [30]:
import rbo

rbo_comparison = pd.DataFrame(columns=['Method_1', 'Method_2', 'RBO_Score'])
methods = features_importance.columns.tolist()
for i in range(len(methods)):
    for j in range(i + 1, len(methods)):
        method_1 = methods[i]
        method_2 = methods[j]
        list_1 = features_importance[method_1].tolist()
        list_2 = features_importance[method_2].tolist()
        rbo_score = rbo.RankingSimilarity(list_1, list_2).rbo(p=0.7, k=8)
        rbo_comparison = pd.concat([rbo_comparison, pd.DataFrame({
            'Method_1': [method_1],
            'Method_2': [method_2],
            'RBO_Score': [rbo_score]
        })], ignore_index=True)
rbo_comparison.sort_values(by='RBO_Score', ascending=False, inplace=True)
rbo_comparison.to_csv('rbo_rank.csv', index=False, sep=';')
rbo_comparison

AssertionError: 